# AlgoCourse Lecture 2: Finding an Alpha
**Imperial Algorithmic Trading Society**

Tonight we follow one trade from idea to execution and use the statistics on real data. There are five short tasks. Each one matches a slide in the lecture: have a go when the slide says **Notebook task**, then we'll show the solution.

**Before you start:** run the first code cell. It loads the price data straight from the course GitHub repo, so there's nothing to download. If you want to keep your answers, use **File → Save a copy in Drive** first.

`PAIR = "shell"` follows Shell against BP. `PAIR = "etf"` follows the US financials ETF (VFH) against four other Vanguard sector ETFs, last year's trade.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, acf

PAIR = "shell"   # "shell" or "etf"

# The data lives in the course GitHub repo, so there's nothing to upload.
REPO = "https://raw.githubusercontent.com/ekaterinaveter/AlgoCourse-2026/main/"
files = {"shell": "shell_bp.csv", "etf": "vanguard_sectors.csv"}
source = files[PAIR] if os.path.exists(files[PAIR]) else REPO + files[PAIR]
prices = pd.read_csv(source, index_col="Date", parse_dates=True).dropna()
if PAIR == "shell":
    Y, X, X_MAIN = "SHEL", ["BP"], "BP"          # Y is what we trade, X is what we hedge it with
else:
    Y, X, X_MAIN = "VFH", ["VDC", "VCR", "VIS", "VGT"], "VIS"
prices.tail()

## Task 1: prices or returns?
Slides: *Puzzle 2* and *Stationarity*.

1. Plot the price of `Y` and of `X_MAIN`.
2. Compute daily **log returns**, $r_t = \ln(P_t / P_{t-1})$, for every column.
3. Compute the correlation between the two **prices**, and between their **returns**.
4. Simulate two *independent* random walks of 500 steps, $x_t = x_{t-1} + \varepsilon_t$, and compare the correlation of their levels with the correlation of their steps. Run it a few times.

Which correlation means something, and why?

In [ ]:
# 1. Plot the prices
prices[[Y, X_MAIN]].plot(figsize=(10, 4), title="Prices")
plt.show()

# 2. Daily log returns for every column
returns = np.log(prices).diff().dropna()

# 3. Correlation of prices and of returns
print("Correlation of prices: ", round(prices[Y].corr(prices[X_MAIN]), 2))
print("Correlation of returns:", round(returns[Y].corr(returns[X_MAIN]), 2))

# 4. Two independent random walks
rng = np.random.default_rng()
steps = rng.normal(0, 1, size=(500, 2))
walks = steps.cumsum(axis=0)
print("Random walks, correlation of levels:", round(np.corrcoef(walks.T)[0, 1], 2))
print("Random walks, correlation of steps: ", round(np.corrcoef(steps.T)[0, 1], 2))

## Task 2: autocorrelation
Slides: *Puzzle 3* and *Reading autocorrelation*.

The autocorrelation at lag $k$ is $\rho(k) = \gamma(k) / \gamma(0)$, where $\gamma(k) = \text{Cov}(x_t, x_{t-k})$.

Using 2015 to 2019 only:
1. Compute $\rho(1), \dots, \rho(10)$ for the daily returns of `Y`. (Try `acf` from statsmodels, or `Series.autocorr(lag=k)`.)
2. Do the same for the **squared** demeaned returns.
3. Plot both as bar charts. What does each one tell you?

In [ ]:
r = returns[Y].loc[:"2019"]

acf_returns = acf(r, nlags=10)[1:]
acf_squared = acf((r - r.mean()) ** 2, nlags=10)[1:]

fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
ax[0].bar(range(1, 11), acf_returns); ax[0].set_title("Returns")
ax[1].bar(range(1, 11), acf_squared); ax[1].set_title("Squared returns")
for a in ax:
    a.axhline(0, color="grey"); a.set_xlabel("lag k")
plt.show()
print("rho(1), returns:", round(acf_returns[0], 2), "  squared returns:", round(acf_squared[0], 2))

## Task 3: build the spread and test it
Slides: *How a statistical test works* and *The spread*. **Fit everything on 2015 to 2019 only.**

1. Regress log `Y` on log `X` (with a constant). The slope(s) are the hedge ratio(s) $\beta$.
2. Build the spread $s_t = \ln Y_t - c - \beta \ln X_t$ for the **whole** period, using the 2015 to 2019 fit.
3. Run the Engle-Granger cointegration test, `coint(...)`, on 2015 to 2019. Print the statistic, the 5% critical value and the p-value.
4. Half-life: regress $\Delta s_t$ on $s_{t-1}$ (with a constant). If the slope is $\gamma$, the half-life is $-\ln 2 / \ln(1+\gamma)$ days.
5. z-score: $z_t = (s_t - \bar s) / \text{sd}(s)$, with the mean and sd from 2015 to 2019. Plot $z$ for 2015 to 2019 with lines at $\pm 2$.

In [ ]:
logp = np.log(prices)
fit = logp.loc[:"2019"]

# 1. Hedge ratio(s)
model = sm.OLS(fit[Y], sm.add_constant(fit[X])).fit()
beta = model.params[X]
print("Hedge ratio(s):"); print(beta.round(2))

# 2. Spread for the whole period
spread = logp[Y] - model.params["const"] - (logp[X] * beta).sum(axis=1)

# 3. Engle-Granger test on 2015 to 2019
stat, pvalue, crit = coint(fit[Y], fit[X])
print(f"Engle-Granger statistic {stat:.2f}, 5% critical value {crit[1]:.2f}, p-value {pvalue:.3f}")

# 4. Half-life
s_fit = spread.loc[:"2019"]
hl_reg = sm.OLS(s_fit.diff().dropna(), sm.add_constant(s_fit.shift(1).dropna())).fit()
gamma = hl_reg.params.iloc[1]
print(f"Half-life: {-np.log(2) / np.log(1 + gamma):.0f} trading days")

# 5. z-score and plot
z = (spread - s_fit.mean()) / s_fit.std()
z.loc[:"2019"].plot(figsize=(10, 4), title="z-score of the spread, 2015 to 2019")
for level in (-2, 0, 2):
    plt.axhline(level, color="red" if level else "grey", linestyle="--" if level else "-")
plt.show()

## Task 4: backtest the rule on 2015 to 2019
Slides: *The trading rule* and *Backtest*.

Rule: if $z > +2$, short the spread (position $-1$); if $z < -2$, buy it ($+1$); close when $z$ crosses back through 0.

1. Write `positions(z)` that returns the position at each date's close.
2. Daily P&L per unit of spread: yesterday's position × today's change in the spread. **Why yesterday's?**
3. Charge costs: each time the position changes by 1, every leg crosses the spread, at `cost_bp` basis points per £1 traded.
4. Divide by the gross exposure, $1 + \sum|\beta|$, to get a return on the money in the trade. Report the annualised Sharpe ratio, maximum drawdown and number of trades.

*(Why no risk-free rate in this Sharpe? The short leg pays for the long leg, so no cash is tied up.)*

In [ ]:
def positions(z, entry=2.0, exit=0.0):
    pos = []
    current = 0
    for zi in z:
        if current == 0 and zi > entry:
            current = -1
        elif current == 0 and zi < -entry:
            current = 1
        elif current == 1 and zi >= exit:
            current = 0
        elif current == -1 and zi <= exit:
            current = 0
        pos.append(current)
    return pd.Series(pos, index=z.index)


def backtest(z, logp, cost_bp=5.0, plot=True):
    pos = positions(z)
    spread_change = logp[Y].diff() - (logp[X].diff() * beta).sum(axis=1)
    gross = 1 + beta.abs().sum()
    pnl = (pos.shift(1) * spread_change).fillna(0)          # trade on yesterday's signal only
    costs = pos.diff().abs().fillna(0) * gross * cost_bp / 1e4
    ret = (pnl - costs) / gross
    cum = ret.cumsum()
    stats = {
        "Sharpe": round(float(ret.mean() / ret.std() * np.sqrt(252)), 2),
        "annualised return": f"{ret.mean() * 252:.1%}",
        "annualised volatility": f"{ret.std() * np.sqrt(252):.1%}",
        "max drawdown": f"{(cum - cum.cummax()).min():.1%}",
        "trades": int(((pos != 0) & (pos.shift(1).fillna(0) == 0)).sum()),
        "share of days in a trade": f"{(pos != 0).mean():.0%}",
    }
    if plot:
        (100 * cum).plot(figsize=(10, 4), title="Cumulative return (%)")
        plt.show()
    return stats


cost = 5.0 if PAIR == "shell" else 1.0
backtest(z.loc[:"2019"], logp.loc[:"2019"], cost_bp=cost)

## Task 5: what happened next?
Slides: *Puzzle 7* and *What happened next*.

Run **exactly the same rule**, with the same hedge ratio, mean and standard deviation from 2015 to 2019, on 2020 onwards. Don't refit anything.

1. Backtest 2020 to today.
2. Plot $z$ for the whole period, 2015 to today.
3. Rerun the Engle-Granger test on 2020 to today. What changed?

In [ ]:
# 1. Same rule, data it has never seen
print(backtest(z.loc["2020":], logp.loc["2020":], cost_bp=cost))

# 2. z-score for the whole period
ax = z.loc[:"2019"].plot(figsize=(10, 4), label="fitted: 2015 to 2019")
z.loc["2020":].plot(ax=ax, color="red", label="traded: 2020 onwards")
for level in (-2, 2):
    ax.axhline(level, color="grey", linestyle="--")
ax.legend(); ax.set_title("z-score of the spread"); plt.show()
print(f"z today: {z.iloc[-1]:.1f}")

# 3. Engle-Granger on 2020 onwards
stat, pvalue, crit = coint(logp.loc["2020":, Y], logp.loc["2020":, X])
print(f"2020 onwards: statistic {stat:.2f}, 5% critical value {crit[1]:.2f}, p-value {pvalue:.2f}")

## Extensions (in your own time)
- **Walk-forward:** refit $\beta$, the mean and the sd every year using only the previous two years, then trade the next year. Does it survive 2020 to today?
- **Overfitting:** try every entry level in 1.0, 1.25, ..., 3.0 and exit level in −0.5, 0, 0.5 on 2015 to 2019. Pick the best. How does it do after 2020?
- **Risk management:** add a stop-loss that closes the trade, and stops trading, once $|z| > 4$. Add a rolling two-year cointegration test and stop opening trades when its p-value goes above 0.10.
- **Switch pair:** set `PAIR = "etf"` at the top and rerun everything. Which trade would you rather have presented?